In [1]:
import json
import random
from PIL import Image
from torch.utils.data import Dataset
import torch
from transformers import CLIPProcessor, CLIPModel
from torch.utils.data import DataLoader
from transformers import CLIPProcessor
import numpy as np
from tqdm import tqdm
from transformers import CLIPModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
class ImageTextDataset(Dataset):
    def __init__(self, jsonl_path, processor, split="train", train_ratio=0.8, val_ratio=0.1, seed=42):
        with open(jsonl_path, "r", encoding="utf-8") as f:
            data = [json.loads(line) for line in f]
        random.seed(seed)
        random.shuffle(data)

        n_train = int(len(data) * train_ratio)
        n_val = int(len(data) * val_ratio)

        if split == "train":
            self.data = data[:n_train]
        elif split == "val":
            self.data = data[n_train:n_train+n_val]
        elif split == "test":
            self.data = data[n_train+n_val:]
        elif split == "all":
            self.data = data

        self.processor = processor

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        image = Image.open(item["image_path"]).convert("RGB")
        caption = item["caption"]
        label = int(item["label"])
        inputs = self.processor(
            text=caption,
            images=image,
            return_tensors="pt",
            padding="max_length",
            truncation=True
        )
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}
        return inputs, label

In [8]:
def extract_embeddings(model, dataloader, device, fine_tune=False):
    model.eval()
    all_embeds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, desc="Extracting embeddings"):
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            outputs = model(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                pixel_values=inputs["pixel_values"]
            )

            # Mean on text and image embeddings
            combined_embeds = (outputs.text_embeds + outputs.image_embeds) / 2

            all_embeds.append(combined_embeds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_embeds = np.vstack(all_embeds)
    all_labels = np.array(all_labels)
    np.savez(f"clip_embeddings{"_ft" if fine_tune else ""}.npz", embeddings=all_embeds, labels=all_labels)

In [ ]:
jsonl_path = "combined_dataset.jsonl"
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
dataset = ImageTextDataset(jsonl_path, processor, split="all")
dataloader = DataLoader(dataset, batch_size=32, shuffle=False)
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [5]:
extract_embeddings(model, dataloader, device, fine_tune=False)

Extracting embeddings: 100%|██████████| 1507/1507 [18:41<00:00,  1.34it/s]


In [6]:
model_ft = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
model_ft.load_state_dict(torch.load("clip_best_model.pth"))

<All keys matched successfully>

In [9]:
extract_embeddings(model_ft, dataloader, device, fine_tune=True)

Extracting embeddings: 100%|██████████| 1507/1507 [10:09<00:00,  2.47it/s]
